In [1]:
# Importações
import pandas as pd
from pandas.plotting import register_matplotlib_converters

register_matplotlib_converters()

import pytz
import MetaTrader5 as mt5

import sys
from pathlib import Path

from datetime import datetime

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
from src.data_handler.provider import YFinanceProvider, MetaTraderProvider
from src.strategies.lstm import LSTMStrategy 
from src.backtest_engine.runner import run_external_simulation

In [2]:
# conecte-se ao MetaTrader 5
if not mt5.initialize():
    print("initialize() failed")
    mt5.shutdown()
 
# consultamos o estado e os parâmetros de conexão
print(mt5.terminal_info())
# obtemos informações sobre a versão do MetaTrader 5
print(mt5.version())

TerminalInfo(community_account=True, community_connection=True, connected=True, dlls_allowed=True, trade_allowed=False, tradeapi_disabled=False, email_enabled=False, ftp_enabled=False, notifications_enabled=False, mqid=False, build=5320, maxbars=100000, codepage=1252, ping_last=52837, community_balance=0.0, retransmission=0.5716933136358471, company='Clear (XP Investimentos CCTVM)', name='Clear Investimentos MT5 Terminal', language='Portuguese (Brazil)', path='C:\\Program Files\\Clear Investimentos MT5 Terminal', data_path='C:\\Users\\User\\AppData\\Roaming\\MetaQuotes\\Terminal\\698B86206820B42976F30D28CAC50412', commondata_path='C:\\Users\\User\\AppData\\Roaming\\MetaQuotes\\Terminal\\Common')
(500, 5320, '26 Sep 2025')


In [3]:
# obtemos o número de instrumentos financeiros
symbols=mt5.symbols_total()
if symbols>0:
    print("Total symbols =",symbols)
else:
    print("Symbols not found")

Total symbols = 60671


In [4]:
selected = mt5.symbol_select("WIN$",True)

if not selected:
    print("Failed to select WIN$")
    mt5.shutdown()
    quit()

# imprimimos o último tick do símbolo WIN$
lasttick=mt5.symbol_info_tick("WIN$")
print(lasttick)
# imprimimos os valores dos campos de tick como uma lista
print("Show symbol_info_tick(\"WIN$\")._asdict():")
symbol_info_tick_dict = mt5.symbol_info_tick("WIN$")._asdict()
for prop in symbol_info_tick_dict:
    print("  {}={}".format(prop, symbol_info_tick_dict[prop]))
 
# concluímos a conexão ao terminal MetaTrader 5
#mt5.shutdown()

Tick(time=1759845959, bid=0.0, ask=0.0, last=141670.0, volume=2, time_msc=1759845959265, flags=1080, volume_real=2.0)
Show symbol_info_tick("WIN$")._asdict():
  time=1759845959
  bid=0.0
  ask=0.0
  last=141670.0
  volume=2
  time_msc=1759845959265
  flags=1080
  volume_real=2.0


In [ ]:
# obtemos informações sobre um instrumento financeiro

range = mt5.copy_rates_range("WIN$", mt5.TIMEFRAME_H1, datetime(2025,10,6,0), datetime(2025,10,7,0))
df_range = pd.DataFrame(range)
df_range['time']=pd.to_datetime(df_range['time'], unit='s')
print(df_range)

                   time      open      high       low     close  tick_volume  \
0   2025-10-06 09:00:00  144860.0  145040.0  144765.0  144810.0       127624   
1   2025-10-06 09:05:00  144810.0  144815.0  144475.0  144540.0       117978   
2   2025-10-06 09:10:00  144540.0  144790.0  144450.0  144750.0       103962   
3   2025-10-06 09:15:00  144755.0  144765.0  144605.0  144675.0        71812   
4   2025-10-06 09:20:00  144670.0  144785.0  144630.0  144670.0        48253   
..                  ...       ...       ...       ...       ...          ...   
108 2025-10-06 18:00:00  144150.0  144160.0  144090.0  144150.0         9571   
109 2025-10-06 18:05:00  144155.0  144195.0  144150.0  144175.0        10054   
110 2025-10-06 18:10:00  144175.0  144220.0  144145.0  144180.0        12351   
111 2025-10-06 18:15:00  144180.0  144200.0  144085.0  144110.0        14470   
112 2025-10-06 18:20:00  144105.0  144120.0  144000.0  144060.0         5780   

     spread  real_volume  
0         1 

In [5]:
# obtemos informações sobre um instrumento financeiro

range = mt5.copy_rates_from_pos("WIN$", mt5.TIMEFRAME_H1, 0, 300)
df_range = pd.DataFrame(range)
df_range['time']=pd.to_datetime(df_range['time'], unit='s')
print(df_range)

                   time      open      high       low     close  tick_volume  \
0   2025-08-26 15:00:00  139890.0  139935.0  139705.0  139830.0       251495   
1   2025-08-26 16:00:00  139835.0  140220.0  139810.0  140075.0       242931   
2   2025-08-26 17:00:00  140075.0  140480.0  140065.0  140430.0       209104   
3   2025-08-26 18:00:00  140425.0  140570.0  140400.0  140435.0        65386   
4   2025-08-27 09:00:00  140100.0  140485.0  139540.0  140210.0       777380   
..                  ...       ...       ...       ...       ...          ...   
295 2025-10-07 10:00:00  143115.0  143300.0  142180.0  142305.0       370028   
296 2025-10-07 11:00:00  142305.0  142355.0  141765.0  142070.0       186117   
297 2025-10-07 12:00:00  142070.0  142260.0  141675.0  141825.0       167380   
298 2025-10-07 13:00:00  141825.0  141935.0  141420.0  141630.0       137844   
299 2025-10-07 14:00:00  141630.0  141720.0  141580.0  141620.0        26519   

     spread  real_volume  
0         1 

In [ ]:
# obtemos ticks de um instrumento financeiro

ticks = mt5.copy_ticks_from("WIN$", datetime(2020,1,1,0), 1000, mt5.COPY_TICKS_ALL)
df_ticks = pd.DataFrame(ticks)
print(df_ticks)

Empty DataFrame
Columns: []
Index: []


In [9]:
# desligamos a conexão com o MetaTrader 5
mt5.shutdown()

True